In [ ]:
import sys, os
print(os.getcwd())
print(sorted(os.listdir()))

In [ ]:
# Google Colab

from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/github/quant_dev/notebooks')
print(sorted(os.listdir()))

In [ ]:
# Add the parent directory to the sys.path
sys.path.append(
    os.path.dirname(
        os.path.dirname(os.path.abspath(__file__))
        if "__file__" in globals().keys()
        else os.getcwd()
    )
)

In [ ]:
from src.tools.basic import *

# ES analysis

In [ ]:
start_date = '2016-01-01'
end_date = dt.datetime.strftime(dt.datetime.today(), "%Y-%m-%d")
trade_d = 250

In [ ]:
# Note
# Dow Jones Industrial Average (DJIA) → ^DJI
# S&P 500 Index → ^GSPC
# Nasdaq Composite Index → ^IXIC
# Russell 2000 Index → ^RUT

stock = 'ES=F'
data = yf.download(stock, start_date, end_date)

In [ ]:
for d in [1, 2, 3, 4, 5, 10, 250]:
  if d == 1:
    data["PCT Change 1"] = data["Close"].pct_change()
    data["PCT Change 1 Av"] = data["PCT Change 1"].rolling(trade_d).mean()
    data["PCT Change 1 STD"] = data["PCT Change 1"].rolling(trade_d).std()

  else:
    data[f"PCT Change {d}"] = data["PCT Change 1"].rolling(d).sum()
    data[f"PCT Change {d} Av"] = data[f"PCT Change {d}"].rolling(trade_d).mean()
    data[f"PCT Change {d} STD"] = data[f"PCT Change {d}"].rolling(trade_d).std()

  data["PCT Change Annualized"] = data["PCT Change 1"].rolling(trade_d).sum()
  data["PCT Change Annualized STD"] = data["PCT Change 1 STD"] * trade_d ** 0.5

pct_change = data["PCT Change 1"]


In [ ]:
data.tail()

In [ ]:
# @ Plots
from matplotlib import pyplot as plt
plt.figure(figsize=(20, 8))
for d in [1, 2, 3, 4, 5, 10]:
  data[f"PCT Change {d}"].plot(kind='line', label=f"PCT Change {d}")

plt.xlim(dt.datetime(2021,1,1),)
plt.title("PCT Change")
plt.legend()
plt.show()

plt.figure(figsize=(20, 8))
for d in [1, 2, 3, 4, 5, 10]:
  data[f"PCT Change {d} Av"].plot(kind='line', label=f"PCT Change {d} Av")

plt.xlim(dt.datetime(2021,1,1),)
plt.title("PCT Change Av")
plt.legend()
plt.show()

plt.figure(figsize=(20, 8))
for d in [1, 2, 3, 4, 5, 10]:
  data[f"PCT Change {d} STD"].plot(kind='line', label=f"PCT Change {d} STD")

data["PCT Change Annualized STD"].plot(kind='line', label="PCT Change Annualized STD")

plt.title("PCT Change STD")
plt.xlim(dt.datetime(2021,1,1),)
plt.legend()
plt.show()

In [ ]:
# @title Close

from matplotlib import pyplot as plt
data['Close'].plot(kind='line', figsize=(20, 8), title='Close')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# @title PCT Change

from matplotlib import pyplot as plt
data['PCT Change 1'].plot(kind='line', figsize=(20, 8), title='PCT Change')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
print(f"Min pct change: {pct_change.min()}")
print(f"Max pct change: {pct_change.max()}")

# Vol and mean

## Annualized

In [ ]:
data["PCT Change Annualized"].iloc[-1]

In [ ]:
data["PCT Change Annualized STD"].iloc[-1]

## Daily

In [ ]:
data.iloc[-1]

In [ ]:
sd = data["PCT Change 1 STD"].iloc[-1]
# sd = pct_change[-250:].std()
sd

In [ ]:
mu = data["PCT Change 1 Av"].iloc[-1]
# mu = pct_change[-250:].mean()
mu

# Historical analysis

In [ ]:
sd_his = data["PCT Change 1"].dropna().std()
# sd_his = pct_change.dropna().std()
sd_his

In [ ]:
# Threshold daily volatility
sd_th = 0.01

In [ ]:
df_his = pd.DataFrame()

for sd in [0.05, 0.045, 0.04, 0.035, 0.03, 0.025, 0.02, 0.015, 0.01, 0.005, 0.002, 0.001, 0.0001]:
  for n_y in [2,5]:
    for n_d in [1,2,3,4,5,10,20,30]:
      df = consecutive_analysis(pct_change, sd, n_d, n_y, return_output=True)
      df_his = pd.concat([df_his, df])
    print()

df_his.reset_index(drop=True, inplace=True)

In [ ]:
data.tail()

In [ ]:
# 1 Day
df = df_his.query("n_days == 1").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# 2 Day
df = df_his.query("n_days == 2").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# 3 Day
df = df_his.query("n_days == 3").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# 4 Day
df = df_his.query("n_days == 4").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# 5 Day
df = df_his.query("n_days == 5").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# Two weeks
df = df_his.query("n_days == 10").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# 1 month
df = df_his.query("n_days == 20").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
# 1.5 Months
df = df_his.query("n_days == 30").copy()
df["prob"] = df["prob"].str.rstrip('%').astype('float') / 100
df.loc[df["prob"] < 0.1]

In [ ]:
p_and_l = {
    1:{"above":[0.15, -0.3], "below": [0.15, -0.3], "exceeding": [0.2, -0.3]},
    5:{"above":[0.2, -0.3], "below": [0.2, -0.3], "exceeding": [0.3, -0.3]},
    10:{"above":[0.2, -0.3], "below": [0.2, -0.3], "exceeding": [0.4, -0.3]},
       }

prob_loss, profit_expect = profit_estimate(
    *sd_and_cond(data, sd_th, 0.02),
    p_and_l, 100)

In [ ]:
p_and_l = {
    1:{"above":[0.15, -1], "below": [0.15, -1], "exceeding": [0.2, -0.9]},
    5:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.3, -0.9]},
    10:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.4, -0.9]},
       }

prob_loss, profit_expect = profit_estimate(
    *sd_and_cond(data, sd_th, 0.02),
    p_and_l, 100)

In [ ]:
p_and_l = {
    1:{"above":[0.15, -1], "below": [0.15, -1], "exceeding": [0.2, -0.9]},
    5:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.4, -0.9]},
    10:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.4, -0.9]},
       }
prob_loss, profit_expect = profit_estimate(
    *sd_and_cond(data, sd_th, 0.02),
    p_and_l, 100)

In [ ]:
p_and_l = {
    1:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.2, -0.9]},
    5:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.45, -0.9]},
    10:{"above":[0.2, -1], "below": [0.2, -1], "exceeding": [0.45, -0.9]},
       }
prob_loss, profit_expect = profit_estimate(
    *sd_and_cond(data, sd_th, 0.02),
    p_and_l, 100)